# 계산 도구를 호출하는 에이전트 — 주피터 따라 하기
이 노트북은 **완성된 코드**입니다. 위에서부터 한 셀씩 실행하며 **일반 함수 → `@tool` → `create_agent`** 순서로, 같은 계산을 사람이 직접 부를 때와 모델이 도구를 요청할 때 무엇이 달라지는지 확인합니다.

- 모든 셀은 ▶ 제공 코드입니다. 빈칸이나 채점 셀은 없습니다.
- 직접 구현하는 과제는 별도 런박스에서 합니다. 같은 폴더의 `바이브코딩_정지손실도구/실습_정지손실도구_바이브코딩.html`을 여세요.

이 노트북은 **독립 Runbox용**입니다. 다른 노트북을 열지 않아도 필요한 코드가 모두 들어 있습니다.

### 0. 실습 환경 확인
Python 버전이 **3.12.x**(3.12로 시작)로 나오면 정상입니다.

In [ ]:
# ▶ 제공 코드 — 실습 환경(커널) 확인
import sys
from importlib.metadata import version

print("Python", sys.version.split()[0], "(기준 3.12.x)")
for name in ["langchain", "langchain-core", "langchain-openai", "langgraph"]:
    print(f"{name:18s}", version(name))

### 1. 설정 셀 — 모델 연결
안내받은 **수업용 API 키**(`API_KEY`)와 **채팅 모델 주소**(`BASE_URL`)를 따옴표 안에 직접 입력하세요. 모델은 `gpt-6-luna`로 정해져 있어 바꾸지 않습니다. 이 실습은 채팅 모델만 쓰므로 임베딩 설정은 없습니다.
- 두 값 중 하나라도 자리표시자로 남아 있으면 이 셀이 모델을 호출하기 전에 멈추고 무엇을 입력할지 알려 줍니다.
- `.env` 파일은 사용하지 않습니다.
- 이 셀은 AI 채팅창이나 AI 코딩 도우미에 **붙여넣지 마세요**. 노트북을 제출·공유하기 전에는 키를 다시 `"여기에_수업용_API_키_입력"`으로 되돌립니다.

In [ ]:
# ▶ 제공 코드 — 설정 셀 (따옴표 안의 값만 바꾸세요)
API_KEY = "여기에_수업용_API_키_입력"
BASE_URL = "여기에_채팅용_BASE_URL_입력"   # 안내받은 채팅 모델 주소(https://...)
MODEL_NAME = "gpt-6-luna"                  # 수업에서 쓰는 채팅·도구 호출 모델(확정값, 바꾸지 않음)

missing = [name for name, value in (("API_KEY", API_KEY), ("BASE_URL", BASE_URL)) if value.startswith("여기에")]
if missing:
    raise ValueError(", ".join(missing) + "에 안내받은 값을 따옴표 안에 입력한 뒤 다시 실행하세요. (모델을 호출하기 전에 멈췄습니다)")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODEL_NAME, api_key=API_KEY, base_url=BASE_URL)
print("모델 준비 완료:", MODEL_NAME)

### 2. 일반 파이썬 함수 직접 실행
시간당 240개를 만드는 라인이 90분 가동하면 240 × 90 ÷ 60 = **360개**입니다. 90분은 1.5시간이므로 240 × 1.5로 검산해도 같습니다. 여기서는 사람이 값을 넣고 함수를 부릅니다.

In [ ]:
# ▶ 제공 코드 — 계산 함수
def expected_output(units_per_hour: int, operating_minutes: int) -> int:
    """시간당 생산량 × 가동 시간(분) ÷ 60 = 예상 생산량(개, 반올림)"""
    return round(units_per_hour * operating_minutes / 60)

print(expected_output(240, 90))   # 360

### 3. 같은 함수를 `@tool`로 감싸기
`@tool`을 붙이면 모델이 읽을 수 있는 **이름·설명·인자 목록**이 생깁니다. 모델은 대화 내용·시스템 지시와 함께 이 정보를 보고 도구를 고릅니다. 그래서 설명과 인자 이름에 **단위(개/시간, 분)** 와 **언제 쓰는지**를 적습니다.

In [ ]:
# ▶ 제공 코드 — 도구로 감싸기
from langchain.tools import tool

@tool
def calculate_production(units_per_hour: int, operating_minutes: int) -> int:
    """라인이 '가동한' 시간 동안 만든 예상 생산량(개)을 계산합니다.
    units_per_hour는 시간당 생산량(개/시간), operating_minutes는 가동 시간(분)입니다.
    정지·고장으로 못 만든 수량에는 사용하지 않습니다."""
    return expected_output(units_per_hour, operating_minutes)

print("이름:", calculate_production.name)
print("설명:", calculate_production.description)
print("인자:", calculate_production.args)

In [ ]:
# ▶ 제공 코드 — 도구를 사람이 직접 호출 (모델 없이)
print(calculate_production.invoke({"units_per_hour": 240, "operating_minutes": 90}))   # 360

### 4. 에이전트에 도구 등록 — 모델은 요청하고, 실행은 에이전트가
실행 기록(`result["messages"]`)에는 네 종류의 메시지가 차례로 쌓입니다.

| 순서 | 메시지 | 누가 만들었나 |
|---|---|---|
| ① | 사용자 질문 (HumanMessage) | 사용자 |
| ② | 도구 요청 — 도구 이름과 인자 (AIMessage의 `tool_calls`) | 모델 (요청만 하고 실행하지 않음) |
| ③ | 도구 반환값 (ToolMessage) | 에이전트가 파이썬 함수를 실행한 결과 |
| ④ | 최종 답변 (AIMessage) | 모델 |

In [ ]:
# ▶ 제공 코드 — 계산 에이전트와 실행 기록 출력 함수
from langchain.agents import create_agent

CALC_RULES = """당신은 제조 라인의 생산 계산 도우미입니다.
수량 계산은 직접 하지 말고 반드시 알맞은 도구를 사용하세요.
계산에 필요한 값(시간당 생산량, 시간)이 대화에 없으면 도구를 부르지 말고 먼저 물어보세요.
답에는 단위 '개'를 붙이세요."""

def show_run(messages):
    """에이전트 실행 기록을 역할별로 나눠 보여 줍니다."""
    for m in messages:
        if m.type == "human":
            print("① 사용자 질문   :", m.content)
        elif m.type == "ai" and m.tool_calls:
            for call in m.tool_calls:
                print("② 모델의 도구 요청:", call["name"], call["args"])
        elif m.type == "tool":
            print("③ 도구 반환값    :", m.name, "→", m.content, "(ToolMessage)")
        elif m.type == "ai":
            print("④ 최종 답변     :", m.content)

calc_agent = create_agent(model=llm, tools=[calculate_production], system_prompt=CALC_RULES)
print("calc_agent 준비 완료")

In [ ]:
# ▶ 제공 코드 — 모델이 도구를 고르고 인자를 채우는 과정
result = calc_agent.invoke(
    {"messages": [{"role": "user", "content": "시간당 240개를 만드는 라인이 90분 가동하면 몇 개를 만들어?"}]}
)
show_run(result["messages"])

### 5. 모델이 ‘60개’라고 답하면 함수를 실행했다고 할 수 있을까?
같은 질문을 도구 없는 모델에 직접 보내고, 에이전트에도 보냅니다. 두 답이 모두 60개여도 **함수를 실제로 실행했다는 증거**는 실행 기록의 도구 요청(②)과 ToolMessage(③)에만 있습니다.

In [ ]:
# ▶ 제공 코드 — 답만 보기 vs 실행 기록 보기
question = "시간당 240개를 만드는 라인이 15분 가동하면 몇 개를 만들어?"

direct = llm.invoke(question)                     # 도구 없이 모델에 직접
print("[모델에 직접] 답:", direct.content)
print("[모델에 직접] 도구 요청:", getattr(direct, "tool_calls", []) or "없음")
print()
run = calc_agent.invoke({"messages": [{"role": "user", "content": question}]})
show_run(run["messages"])
tool_messages = [m for m in run["messages"] if m.type == "tool"]
print("\n[에이전트] ToolMessage 수:", len(tool_messages), "→ 함수가 실제로 실행된 횟수")

### 6. 필요한 값이 없으면?
실행 전에 예상해 보세요. 이 에이전트에는 대화 기록 저장소가 없으므로 바로 위 질문의 240을 이어받지 않습니다. 도구 요청 없이 시간당 생산량을 되묻는지 확인합니다.

In [ ]:
# ▶ 제공 코드 — 필요한 값이 없는 질문
result = calc_agent.invoke(
    {"messages": [{"role": "user", "content": "그 라인이 30분 가동하면 몇 개야?"}]}
)
show_run(result["messages"])

### 정리
| 단계 | 누가 함수를 부르나 | 실행 증거 |
|---|---|---|
| 일반 함수 `expected_output(240, 90)` | 사람 | 반환값 360 |
| 도구 `calculate_production.invoke({...})` | 사람 | 반환값 360 |
| 에이전트 `calc_agent.invoke(...)` | 모델이 요청 → 에이전트가 실행 | 도구 요청(이름·인자) + ToolMessage |

**이어서 할 일** — 런박스 과제에서 **정지 손실 도구**를 추가하고, 답이 같은 두 질문(15분 가동 / 15분 정지)에서 실제로 호출된 도구 이름·인자·반환값을 대조합니다. 안내: `바이브코딩_정지손실도구/실습_정지손실도구_바이브코딩.html`